In [1]:
# run cell to install prokbert lib (if necessary)
# !pip install git+https://github.com/nbrg-ppcu/prokbert.git@development --quiet

In [ ]:
import os

import datasets
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, balanced_accuracy_score
from transformers import Trainer, TrainingArguments

from prokbert.tokenizer import LCATokenizer
from prokbert.data_collator import SegmentDataCollator
from prokbert.models import ProkBertForSequenceClassification
from prokbert.sequence_dataset import SequenceDataset, EmbeddingDataset
from prokbert.dataset import RandomSegmentDataset, ContiguousSegmentDataset

In [3]:
os.environ["WANDB_PROJECT"]=""
os.environ["HF_HOME"] = ""

In [17]:
base_dir = "./"

### Dataset

In [12]:
train_ds = datasets.load_dataset("neuralbioinfo/PhaStyle-SequenceDB", split="BACPHLIP_TRAINING")
test_ds  = datasets.load_dataset("neuralbioinfo/PhaStyle-SequenceDB", split="ESCHERICHIA")

In [13]:
print(f"train examples: {len(train_ds)}, columns: {train_ds.column_names}")
print(f"test examples: {len(test_ds)}, columns: {test_ds.column_names}")

train examples: 1798, columns: ['sequence_id', 'dataset', 'class_label', 'y', 'source_fasta_id', 'orientation', 'test_fastaid', 'L_seq', 'source_description', 'Genome type', 'Family', 'subds_seqid', 'sequence', '__index_level_0__']
test examples: 394, columns: ['sequence_id', 'dataset', 'class_label', 'y', 'source_fasta_id', 'orientation', 'test_fastaid', 'L_seq', 'source_description', 'Genome type', 'Family', 'subds_seqid', 'sequence', '__index_level_0__']


In [14]:
cols = [
    "dataset", "class_label", "test_fastaid", "L_seq", "source_description",
    "Genome type", "Family", "subds_seqid", "__index_level_0__"
]
train_ds = train_ds.remove_columns(cols)
test_ds = test_ds.remove_columns(cols)

In [15]:
train_ds = train_ds.rename_column("y", "label")
train_ds = train_ds.rename_column("source_fasta_id", "contig_id")

test_ds = test_ds.rename_column("y", "label")
test_ds = test_ds.rename_column("source_fasta_id", "contig_id")

In [16]:
# quick look at first example
# train_ds[0]

In [18]:
train_save_dir = f"{base_dir}/example-train-dataset/"
test_save_dir = f"{base_dir}/example-test-dataset/"

In [ ]:
train_ds.save_to_disk(train_save_dir)
test_ds.save_to_disk(test_save_dir)

### Sequence Dataset

In [20]:
train_seq_dataset_save_dir = f"{base_dir}/example-train-sequence-dataset"

train_seq_ds = SequenceDataset()
train_seq_ds.create_dataset([train_save_dir], save_dir=train_seq_dataset_save_dir)

print("Total length:", len(train_seq_ds.sequence))
print(f"Total contigs: {len(train_seq_ds.metadata)}")

Loaded 1798 contigs from .//example-train-dataset/.
Converted 1798 contigs to Contig format.
hello there
Total length: 116680168
Total contigs: 1798


In [21]:
pd.DataFrame(train_seq_ds.metadata).head()

,genome_id,contig_id,sequence_id,coordinate,orientation,description,label
0,None,PHAA0000323,0,"(0, 53124)",forward,None,0
1,None,PHAA0000323,1,"(53124, 106248)",reverse,None,0
2,None,PHAA0000761,2,"(106248, 177813)",forward,None,1
3,None,PHAA0000761,3,"(177813, 249378)",reverse,None,1
4,None,PHAA0000840,4,"(249378, 302774)",forward,None,0


In [22]:
test_seq_dataset_save_dir = f"{base_dir}/example-test-sequence-dataset"

test_seq_ds = SequenceDataset()
test_seq_ds.create_dataset([test_save_dir], save_dir=test_seq_dataset_save_dir)

print("Total length:", len(test_seq_ds.sequence))
print(f"Total contigs: {len(test_seq_ds.metadata)}")

Loaded 394 contigs from .//example-test-dataset/.
Converted 394 contigs to Contig format.
hello there
Total length: 22399782
Total contigs: 394


In [23]:
pd.DataFrame(test_seq_ds.metadata).head()

,genome_id,contig_id,sequence_id,coordinate,orientation,description,label
0,None,409_P1,0,"(0, 40139)",forward,None,1
1,None,409_P1,1,"(40139, 80278)",reverse,None,1
2,None,409_P3,2,"(80278, 125926)",forward,None,1
3,None,409_P3,3,"(125926, 171574)",reverse,None,1
4,None,409_P5,4,"(171574, 217222)",forward,None,1


In [25]:
# quick check
# for i, meta in enumerate(train_seq_ds.metadata):
#     start, end = meta["coordinate"]
#     print(f"{i:3}: {meta['contig_id']:12} [{start:3}, {end:3})  {train_seq_ds.sequence[start:end]}")
#     if i >= 5: break

### Embedding dataset

#### Create embeddings

To create embeddings for the train dataset, run the following script:

```txt
srun --job-name=phastyle-sequence-db-train-create-embedding-with-mini-long \
     --nodes=1 \
     --nodelist=bolka \
     --ntasks=4 \
     --cpus-per-task=4 \
     --gpus-per-task=1 \
     --mem=100G \
     --time=7-00:00:00 \
     --partition=hi \
     --output=/home/username/logs/%x_%j.out \
     --error=/home/username/logs/%x_%j.err \
     python3 scripts/embed_dataset.py \
        --model neuralbioinfo/prokbert-mini-long \
        --input-dir <base_dir>/example-train-sequence-dataset/ \
        --output-dir <base_dir>/example-train-sequence-dataset/ \
        --seq-len 2000 \
        --pooling-len 100 \
        --batch-size 32 \
        --kmer 6 \
        --shift 2 \
        --pooling-strategy mean \
        --verbose
```

Once all shards have finished, run the concat step once (no GPU needed):

```txt
srun --job-name=concat-embeddings \
     --nodes=1 \
     --cpus-per-task=4 \
     --mem=150G \
     --partition=<partition> \
     --output=<log-dir>/%x_%j_%t.out \
     --error=<log-dir>/%x_%j_%t.err \
     python3 scripts/embed_dataset.py \
        --model neuralbioinfo/prokbert-mini-long \
        --input-dir <data-dir>/example-train-sequence-dataset \
        --output-dir <data-dir>/example-train-sequence-dataset \
        --seq-len 2000 --pooling-len 100 --kmer 6 --shift 2 --pooling-strategy mean \
        --concat-only
```

To create embeddings for the test dataset, run the following script:

```txt
srun --job-name=phastyle-sequence-db-test-create-embedding-with-mini-long \
     --nodes=1 \
     --nodelist=bolka \
     --ntasks=4 \
     --cpus-per-task=4 \
     --gpus-per-task=1 \
     --mem=100G \
     --time=7-00:00:00 \
     --partition=hi \
     --output=/home/username/logs/%x_%j.out \
     --error=/home/username/logs/%x_%j.err \
     python3 scripts/embed_dataset.py \
        --model neuralbioinfo/prokbert-mini-long \
        --input-dir <base_dir>/example-test-sequence-dataset/ \
        --output-dir <base_dir>/example-test-sequence-dataset/ \
        --seq-len 2000 \
        --pooling-len 100 \
        --batch-size 32 \
        --kmer 6 \
        --shift 2 \
        --pooling-strategy mean \
        --verbose
```

Once all shards have finished, run the concat step once (no GPU needed):

```txt
srun --job-name=concat-embeddings \
     --nodes=1 \
     --cpus-per-task=4 \
     --mem=150G \
     --partition=<partition> \
     --output=<log-dir>/%x_%j_%t.out \
     --error=<log-dir>/%x_%j_%t.err \
     python3 scripts/embed_dataset.py \
        --model neuralbioinfo/prokbert-mini-long \
        --input-dir <data-dir>/example-test-sequence-dataset \
        --output-dir <data-dir>/example-test-sequence-dataset \
        --seq-len 2000 --pooling-len 100 --kmer 6 --shift 2 --pooling-strategy mean \
        --concat-only
```

#### Load dataset

In [30]:
train_seq_dataset_save_dir = f"{base_dir}/example-train-sequence-dataset"
test_seq_dataset_save_dir = f"{base_dir}/example-test-sequence-dataset"

In [31]:
emb_ds_train = EmbeddingDataset(
    sequence_dataset=train_seq_ds,
    config_path=f"{train_seq_dataset_save_dir}/config.yaml",
    embedding_file=f"{train_seq_dataset_save_dir}/embeddings_forward.pt",
    seq_tokens_file=f"{train_seq_dataset_save_dir}/seq_tokens.pt",
)

In [32]:
emb_ds_test = EmbeddingDataset(
    sequence_dataset=test_seq_ds,
    config_path=f"{test_seq_dataset_save_dir}/config.yaml",
    embedding_file=f"{test_seq_dataset_save_dir}/embeddings_forward.pt",
    seq_tokens_file=f"{test_seq_dataset_save_dir}/seq_tokens.pt",
)

In [33]:
emb_ds_train.config, emb_ds_test.config

({'tokenizer': 'LCATokenizer',
  'kmer': 6,
  'shift': 2,
  'model': 'neuralbioinfo/prokbert-mini-long',
  'pooling_strategy': 'mean',
  'pooling_length': 100,
  'special_tokens': []},
 {'tokenizer': 'LCATokenizer',
  'kmer': 6,
  'shift': 2,
  'model': 'neuralbioinfo/prokbert-mini-long',
  'pooling_strategy': 'mean',
  'pooling_length': 100,
  'special_tokens': []})

## Data Loader and Sampling 

In [ ]:
# parameters
min_segment_length = 1000
max_segment_length = 2000

kmer = 6
shift = 2
add_special_tokens = False
return_metadata = False

batch_size = 16

### Sequence Segment Dataset

In [35]:
train_dataset = RandomSegmentDataset(
    sequence_dataset=train_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    return_sequence=True,
    coverage=2,
    seed=42,
)
test_dataset = ContiguousSegmentDataset(
    sequence_dataset=test_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    return_sequence=True,
)

In [ ]:
for i, segment in enumerate(iter(train_dataset)):
    if i >= 3: break
    print(segment)

### Embedding Segment Dataset

In [38]:
train_dataset = RandomSegmentDataset(
    sequence_dataset=train_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    embedding_dataset=emb_ds_train,
    return_sequence=False,
    return_embedding=True,
    coverage=2,
    seed=42,
)
test_dataset = ContiguousSegmentDataset(
    sequence_dataset=test_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    embedding_dataset=emb_ds_test,
    return_sequence=False,
    return_embedding=True
)

In [ ]:
for i, segment in enumerate(iter(train_dataset)):
    if i >= 3: break
    print(segment)

In [40]:
# quick check of the first few labels in the train dataset with and without shuffling
from itertools import islice
for shuffle in (False, True):
    train_dataset.shuffle = shuffle
    labels = [seg["label"] for seg in islice(train_dataset, 300)]
    print(f"shuffle={shuffle}: {''.join(str(l) for l in labels[:100])}")

shuffle=False: 0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
shuffle=True: 1011001010101010101111110111000000101000011000110001010110010110001001111101010010110010101110111111


### Data collator

In [ ]:
tokenizer = LCATokenizer(kmer = kmer, shift = shift)
collator = SegmentDataCollator(
    tokenizer = tokenizer,
    add_special_tokens = add_special_tokens,
    return_metadata = return_metadata,
    has_embedding=True, # comment out if not using embeddings (or set to false)
)
loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collator)

In [ ]:
batch = next(iter(loader))

# print(batch["input_ids"].shape)   # uncomment if not using embeddings
print(batch["inputs_embeds"].shape) # comment if not using embeddings
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

## Model training

In [52]:
model_name = "neuralbioinfo/prokbert-mini-long"
num_labels = len(train_ds.unique("label")) # 2

In [57]:
model = ProkBertForSequenceClassification.from_pretrained(model_name, trust_remote_code=True)

In [62]:
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"model parameters: {params / 1e6: .2f}M")

model parameters:  26.43M


In [ ]:
model(**batch)

In [65]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):   # some models also return hidden states
        logits = logits[0]
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
        "mcc": matthews_corrcoef(labels, preds),
    }

In [66]:
training_args = TrainingArguments(
    output_dir=f"{base_dir}/test-phastyle-sequence-db-embedding-db-mini-long",
    max_steps=20_000,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 4,
    learning_rate=5e-5,
    warmup_steps=0.1,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    dataloader_num_workers=4,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to="wandb",
    run_name="test-phastyle-sequence-db-embedding-db-mini-long-head",
    seed=42,
)

In [67]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Run training as a SLURM job using srun and torchrun

```txt
srun --job-name=test-phastyle-sequence-db-embedding-db-mini-long \
     --nodes=1 \
     --nodelist=nodename \
     --ntasks-per-node=1 \
     --cpus-per-task=8 \
     --gres=gpu:1 \
     --mem=100G \
     --time=7-00:00:00 \
     --partition=hi \
     --output=logs/training/%x_%j.out \
     --error=logs/training/%x_%j.err \
     torchrun --standalone --nproc_per_node=1 examples/phastyle_training.py
```